In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

/Users/benlockhart/Downloads/BenFlix/.venv/lib/python3.14/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
df = yf.download("^GSPC", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']

df1 = yf.download("SPXL", period="10y", interval="1d")
df1.columns = df1.columns.get_level_values(0)
df1 = df1.reset_index()
df1['Date'] = pd.to_datetime(df1['Date'])
del df1['Volume']

df2 = yf.download("SPXS", period="10y", interval="1d")
df2.columns = df2.columns.get_level_values(0)
df2 = df2.reset_index()
df2['Date'] = pd.to_datetime(df2['Date'])
del df2['Volume']


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [3]:
df3 = df1[['Date']].copy()
df3['Open'] = df1['Open'] / df2['Close']
df3['High'] = df1['High'] / df2['Close']
df3['Low'] = df1['Low'] / df2['Close']
df3['Close'] = df1['Close'] / df2['Close']

df3


Price,Date,Open,High,Low,Close
0,2016-06-06,0.003380,0.003430,0.003372,0.003410
1,2016-06-07,0.003438,0.003475,0.003435,0.003442
2,2016-06-08,0.003485,0.003516,0.003478,0.003508
3,2016-06-09,0.003449,0.003485,0.003434,0.003478
4,2016-06-10,0.003307,0.003328,0.003260,0.003292
...,...,...,...,...,...
2510,2026-06-01,11.022162,11.232893,10.992224,11.153188
2511,2026-06-02,11.131147,11.280249,11.120999,11.242388
2512,2026-06-03,10.939648,10.968678,10.744843,10.772727
2513,2026-06-04,10.799459,11.068393,10.772025,11.014683


In [4]:
def STO(data, n1, n2, n3):
    hi = data['High'].rolling(n1).max()
    lo = data['Low'].rolling(n1).min()
    k = ((data['Close'] - lo) / (hi - lo)) * 100
    kx = k.rolling(n3).mean()
    d = kx.rolling(n2).mean()
    return kx, d

df3['K'], df3['D'] = STO(df3, 10, 4, 4)
df3.dropna(inplace=True)
df3


Price,Date,Open,High,Low,Close,K,D
15,2016-06-27,0.002584,0.002586,0.002470,0.002500,36.550702,42.597777
16,2016-06-28,0.002726,0.002792,0.002710,0.002782,34.167030,43.003376
17,2016-06-29,0.003014,0.003102,0.003008,0.003091,24.797564,35.561391
18,2016-06-30,0.003250,0.003361,0.003221,0.003354,46.205023,35.430080
19,2016-07-01,0.003370,0.003424,0.003368,0.003390,68.441430,43.402762
...,...,...,...,...,...,...,...
2510,2026-06-01,11.022162,11.232893,10.992224,11.153188,96.587191,91.374968
2511,2026-06-02,11.131147,11.280249,11.120999,11.242388,97.386522,95.391483
2512,2026-06-03,10.939648,10.968678,10.744843,10.772727,90.556562,95.226377
2513,2026-06-04,10.799459,11.068393,10.772025,11.014683,87.345455,92.968932


In [5]:
del df3['Open'], df3['High'], df3['Low'], df3['Close']
df = df.merge(df3, on='Date', how='left')
df.dropna(inplace=True)
df

Price,Date,Close,High,Low,Open,K,D
15,2016-06-27,2000.540039,2031.449951,1991.680054,2031.449951,36.550702,42.597777
16,2016-06-28,2036.089966,2036.089966,2006.670044,2006.670044,34.167030,43.003376
17,2016-06-29,2070.770020,2073.129883,2042.689941,2042.689941,24.797564,35.561391
18,2016-06-30,2098.860107,2098.939941,2070.000000,2073.169922,46.205023,35.430080
19,2016-07-01,2102.949951,2108.709961,2097.899902,2099.340088,68.441430,43.402762
...,...,...,...,...,...,...,...
2510,2026-06-01,7599.959961,7617.660156,7562.609863,7582.290039,96.587191,91.374968
2511,2026-06-02,7609.779785,7620.899902,7582.990234,7595.399902,97.386522,95.391483
2512,2026-06-03,7553.680176,7605.350098,7551.220215,7605.310059,90.556562,95.226377
2513,2026-06-04,7584.310059,7598.189941,7516.540039,7516.540039,87.345455,92.968932


In [6]:
def signal(data):
    signal = [0] * len(data)
    for i in range(2,len(data)):
        if (data.D.iloc[i] > data.D.iloc[i-1]) and (data.D.iloc[i-1] < data.D.iloc[i-2]):
            signal[i] = 1
        elif ():
            signal[i] = 2
        else:
            signal[i] = 0
        data["signal"] = signal
        
signal(df)


In [7]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

print(df['signal'].value_counts())
df.shape

signal
0    2287
1     213
Name: count, dtype: int64


(2500, 9)

In [8]:
df.set_index('Date', inplace=True)

In [9]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.K, 
                         line=dict(color='red', width=1),
                         name='%K'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.D, 
                         line=dict(color='blue', width=2),
                         name='%D'),
                         row=2, col=1)

fig.add_hline(y=80, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=50, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=20, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.update_layout(
    annotations=[dict(text=" Ratio Oscillator ",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=1.00,
                    y=0.3,
                    showarrow=False)])

fig.update_layout(autosize=False, width=1100, height=800, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [10]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, sl=0.95*price, tp=1.035*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2499 [00:00<?, ?bar/s]

Start                     2016-06-27 00:00:00
End                       2026-06-05 00:00:00
Duration                   3630 days 00:00:00
Exposure Time [%]                       70.88
Equity Final [$]                 384662.23831
Equity Peak [$]                   396190.2554
Commissions [$]                   20351.79079
Return [%]                          284.66224
Buy & Hold Return [%]               269.08735
Return (Ann.) [%]                    14.54497
Volatility (Ann.) [%]                 13.0576
CAGR [%]                              9.80373
Sharpe Ratio                          1.11391
Sortino Ratio                         1.83445
Calmar Ratio                          1.08879
Alpha [%]                           174.25922
Beta                                  0.41029
Max. Drawdown [%]                   -13.35886
Avg. Drawdown [%]                    -1.55346
Max. Drawdown Duration      201 days 00:00:00
Avg. Drawdown Duration       17 days 00:00:00
# Trades                          

In [11]:
trades = stats['_trades']

trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [12]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='% / Trade',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()